# 📘 Logging & Monitoring for Data Engineering
## Databricks Notebook — Production Observability

**What you'll learn:**
- Python `logging` module for pipelines
- Structured logging (JSON format) with correlation IDs
- Pipeline monitoring: metrics, SLAs, anomaly detection
- Alerting patterns with cooldown logic
- Databricks-specific logging
- The three pillars: Logs + Metrics + Traces

**Prerequisites:** Notebooks 01-12

**Key rules:**
- Always use named loggers, never root logger
- Never log sensitive data (PII, passwords, keys)
- Include correlation_id for tracing across stages

---

---
# 📗 Section 1: Why Logging Matters in DE

**Without logging:** "The pipeline failed at 3 AM. What happened? No idea."

**With logging:** "Stage 'transform_orders' failed at 03:14:22. Input: 50,000 rows.
Error: NullPointerException on column 'customer_id' at row 34,521. Batch: batch_20240615_003."

**Use cases:**
1. **Debugging:** Reconstruct what happened during a failure
2. **Audit:** Prove data was processed correctly (compliance)
3. **Performance:** Identify bottlenecks (which stage is slow?)
4. **Alerting:** Notify team before users notice issues
5. **Post-mortem:** Reconstruct event sequence after incidents

---
# 📗 Section 2: Python logging Module

## Logger → Handler → Formatter

```
Logger (decides what to log)
  └── Handler (decides where to send it)
        └── Formatter (decides how it looks)

Levels: DEBUG < INFO < WARNING < ERROR < CRITICAL
```

In [0]:
import logging

# Create a named logger (NEVER use root logger in production)
logger = logging.getLogger("pipeline.demo")
logger.setLevel(logging.DEBUG)

# Console handler with formatting
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S"
)
console_handler.setFormatter(formatter)

# Avoid duplicate handlers in notebooks (re-run safe)
logger.handlers = [console_handler]

# Test all levels
logger.debug("This is DEBUG (won't show — handler level is INFO)")
logger.info("Pipeline started")
logger.warning("Data quality issue: 5% null emails")
logger.error("Failed to connect to source database")
logger.critical("Pipeline aborted — data corruption detected")

---
# 📗 Section 3: Structured Logging (JSON Format)

**WHY:** JSON logs are machine-parseable. You can query them, aggregate them,
and feed them into monitoring tools (Datadog, Splunk, CloudWatch).

In [0]:
import logging
import json as json_lib
import uuid
from datetime import datetime

class JSONFormatter(logging.Formatter):
    """Format log records as JSON for machine parsing."""
    
    def format(self, record):
        log_entry = {
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        # Add extra fields if present
        for key in ["pipeline", "stage", "batch_id", "correlation_id", "record_count", "duration_ms"]:
            if hasattr(record, key):
                log_entry[key] = getattr(record, key)
        
        if record.exc_info:
            log_entry["exception"] = self.formatException(record.exc_info)
        
        return json_lib.dumps(log_entry)

# Create structured logger
struct_logger = logging.getLogger("pipeline.structured")
struct_logger.setLevel(logging.DEBUG)
json_handler = logging.StreamHandler()
json_handler.setFormatter(JSONFormatter())
struct_logger.handlers = [json_handler]

# Log with context
correlation_id = str(uuid.uuid4())[:8]
struct_logger.info("Pipeline started", extra={
    "pipeline": "orders_etl",
    "batch_id": "batch_20240615",
    "correlation_id": correlation_id
})
struct_logger.info("Transform complete", extra={
    "pipeline": "orders_etl",
    "stage": "silver_transform",
    "record_count": 9847,
    "correlation_id": correlation_id,
    "duration_ms": 3420
})

---
# 📗 Section 4: Logging Patterns for Pipelines

In [0]:
import time
import logging
from contextlib import contextmanager

# Pipeline stage context manager (auto-logs start/end/duration)
@contextmanager
def pipeline_stage(logger, stage_name, **extra):
    """Context manager that logs stage start, end, and duration."""
    start = time.time()
    logger.info(f"Stage '{stage_name}' started", extra={**extra, "stage": stage_name})
    try:
        yield
        duration = (time.time() - start) * 1000
        logger.info(f"Stage '{stage_name}' completed", extra={
            **extra, "stage": stage_name, "duration_ms": int(duration)
        })
    except Exception as e:
        duration = (time.time() - start) * 1000
        logger.error(f"Stage '{stage_name}' FAILED: {e}", extra={
            **extra, "stage": stage_name, "duration_ms": int(duration)
        })
        raise

# Use it in a pipeline
pipe_logger = logging.getLogger("pipeline.orders")
pipe_logger.setLevel(logging.DEBUG)
pipe_logger.handlers = [json_handler]

ctx = {"pipeline": "orders_etl", "correlation_id": "abc123", "batch_id": "batch_001"}

with pipeline_stage(pipe_logger, "extract", **ctx):
    orders = spark.table("techmart_dw.orders")
    count = orders.count()
    pipe_logger.info(f"Extracted {count:,} rows", extra={**ctx, "record_count": count})

with pipeline_stage(pipe_logger, "transform", **ctx):
    filtered = orders.filter("status = 'completed'")
    out_count = filtered.count()
    pipe_logger.info(f"Filtered: {count:,} → {out_count:,}", extra={
        **ctx, "record_count": out_count, "stage": "transform"
    })

---
# 📗 Section 5: Log Levels Strategy

| Level | When to Use | Example |
|---|---|---|
| DEBUG | Detailed diagnostics (dev only) | "Processing row 34521, customer_id=None" |
| INFO | Normal progress | "Extracted 50,000 rows in 3.4s" |
| WARNING | Unexpected but non-fatal | "5% null emails (threshold: 10%)" |
| ERROR | Failure in one component | "Failed to write to table X, retrying..." |
| CRITICAL | Pipeline must stop | "Data corruption detected, aborting" |

In [0]:
# ============================================
# 🎯 YOUR TURN: Assign Log Levels
# ============================================
# For each scenario, decide the appropriate log level:
#
# 1. Pipeline started successfully → ?
# 2. 3 out of 50,000 records failed validation → ?
# 3. Database connection timed out (will retry) → ?
# 4. All retries exhausted, cannot connect → ?
# 5. Record count dropped 80% from yesterday → ?
# 6. Processing batch 15 of 20 → ?
# 7. Duplicate primary keys detected in source → ?
# 8. Pipeline finished in 45 minutes (SLA is 60 min) → ?
#
# Write your answers below:


In [0]:
# ============================================
# ✅ SOLUTION
# ============================================
# 1. Pipeline started → INFO
# 2. 3/50000 failed validation → WARNING (below threshold, but notable)
# 3. DB timeout, will retry → WARNING
# 4. All retries exhausted → ERROR (or CRITICAL if pipeline must stop)
# 5. Record count dropped 80% → WARNING (anomaly, investigate)
# 6. Processing batch 15/20 → INFO (progress tracking)
# 7. Duplicate PKs in source → ERROR (data integrity issue)
# 8. Finished in 45 min (SLA 60) → INFO (success)
print("See comments above for answers")

---
# 📗 Section 6: Pipeline Monitoring & Metrics

In [0]:
import time
from datetime import datetime
import pandas as pd

class PipelineMetrics:
    """Collect and store pipeline execution metrics."""
    
    def __init__(self, pipeline_name, batch_id):
        self.pipeline_name = pipeline_name
        self.batch_id = batch_id
        self.start_time = time.time()
        self.metrics = []
    
    def record(self, metric_name, value, stage=None, dimensions=None):
        self.metrics.append({
            "pipeline_name": self.pipeline_name,
            "batch_id": self.batch_id,
            "metric_name": metric_name,
            "metric_value": float(value),
            "stage": stage,
            "dimensions": str(dimensions or {}),
            "recorded_at": datetime.utcnow().isoformat()
        })
    
    def summary(self):
        total_duration = time.time() - self.start_time
        self.record("total_duration_seconds", total_duration)
        
        print(f"\n{'='*50}")
        print(f"PIPELINE METRICS: {self.pipeline_name}")
        print(f"{'='*50}")
        print(f"Batch: {self.batch_id}")
        print(f"Duration: {total_duration:.1f}s")
        print(f"Metrics collected: {len(self.metrics)}")
        for m in self.metrics:
            print(f"  {m['metric_name']}: {m['metric_value']:.2f} ({m['stage'] or 'overall'})")
        print(f"{'='*50}")
    
    def write_to_delta(self, table_name="techmart_dw.pipeline_metrics"):
        pdf = pd.DataFrame(self.metrics)
        spark.createDataFrame(pdf).write.format("delta").mode("append").saveAsTable(table_name)
        print(f"  ✅ Metrics written to {table_name}")

# Use in a pipeline
metrics = PipelineMetrics("orders_etl", "batch_20240615_001")

# Extract stage
start = time.time()
orders = spark.table("techmart_dw.orders")
extract_count = orders.count()
metrics.record("rows_extracted", extract_count, stage="extract")
metrics.record("extract_duration_ms", (time.time()-start)*1000, stage="extract")

# Transform stage
start = time.time()
completed = orders.filter("status = 'completed'")
transform_count = completed.count()
metrics.record("rows_transformed", transform_count, stage="transform")
metrics.record("rows_filtered", extract_count - transform_count, stage="transform")
metrics.record("transform_duration_ms", (time.time()-start)*1000, stage="transform")

# Error rate
metrics.record("error_rate_pct", (extract_count - transform_count) / extract_count * 100)

metrics.summary()
metrics.write_to_delta()

---
# 📗 Section 7: Alerting Patterns

In [0]:
from datetime import datetime, timedelta

class AlertManager:
    """Simple alert system with cooldown to prevent alert fatigue."""
    
    def __init__(self, cooldown_minutes=15):
        self.cooldown_minutes = cooldown_minutes
        self.last_alert_time = {}  # alert_key → last fired time
        self.alerts_fired = []
    
    def should_alert(self, alert_key):
        """Check if enough time has passed since last alert of this type."""
        if alert_key not in self.last_alert_time:
            return True
        elapsed = (datetime.now() - self.last_alert_time[alert_key]).total_seconds() / 60
        return elapsed >= self.cooldown_minutes
    
    def fire(self, severity, message, alert_key=None):
        """Fire an alert (with cooldown check)."""
        key = alert_key or message
        
        if not self.should_alert(key):
            return  # Suppressed (cooldown)
        
        self.last_alert_time[key] = datetime.now()
        alert = {"severity": severity, "message": message, "fired_at": datetime.now().isoformat()}
        self.alerts_fired.append(alert)
        
        icon = {"CRITICAL": "🔴", "ERROR": "🟠", "WARNING": "🟡"}.get(severity, "ℹ️")
        print(f"  {icon} [{severity}] {message}")
    
    def check_threshold(self, metric_name, value, warning_threshold, error_threshold):
        """Check a metric against thresholds."""
        if value >= error_threshold:
            self.fire("ERROR", f"{metric_name} = {value} (threshold: {error_threshold})", alert_key=metric_name)
        elif value >= warning_threshold:
            self.fire("WARNING", f"{metric_name} = {value} (threshold: {warning_threshold})", alert_key=metric_name)

# Demo
alerts = AlertManager(cooldown_minutes=5)

print("Alert checks:")
alerts.check_threshold("error_rate_pct", 2.5, warning_threshold=5, error_threshold=10)
alerts.check_threshold("error_rate_pct", 7.0, warning_threshold=5, error_threshold=10)
alerts.check_threshold("error_rate_pct", 15.0, warning_threshold=5, error_threshold=10)
alerts.check_threshold("duration_minutes", 45, warning_threshold=50, error_threshold=60)
alerts.check_threshold("record_count_drop_pct", 55, warning_threshold=30, error_threshold=50)

print(f"\nTotal alerts fired: {len(alerts.alerts_fired)}")

---
# 🚀 Section 8: Mini Project — Production Pipeline Logger

In [0]:
import logging
import json as json_lib
import uuid
import time
from datetime import datetime
from contextlib import contextmanager
import pandas as pd

class ProductionPipelineLogger:
    """Production-grade pipeline logger with structured output and metrics."""
    
    def __init__(self, pipeline_name, batch_id=None):
        self.pipeline_name = pipeline_name
        self.batch_id = batch_id or f"batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        self.correlation_id = str(uuid.uuid4())[:8]
        self.start_time = time.time()
        self.stage_metrics = []
        self.log_entries = []
        
        # Setup logger
        self.logger = logging.getLogger(f"pipeline.{pipeline_name}")
        self.logger.setLevel(logging.DEBUG)
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-5s | %(message)s", datefmt="%H:%M:%S"))
        self.logger.handlers = [handler]
    
    def _log(self, level, message, **kwargs):
        entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": level,
            "pipeline": self.pipeline_name,
            "batch_id": self.batch_id,
            "correlation_id": self.correlation_id,
            "message": message,
            **kwargs
        }
        self.log_entries.append(entry)
        getattr(self.logger, level.lower())(f"[{self.correlation_id}] {message}")
    
    @contextmanager
    def stage(self, stage_name):
        """Context manager for pipeline stages."""
        self._log("INFO", f"Stage '{stage_name}' started", stage=stage_name)
        stage_start = time.time()
        stage_data = {"stage": stage_name, "rows_in": 0, "rows_out": 0}
        
        try:
            yield stage_data
            duration_ms = int((time.time() - stage_start) * 1000)
            stage_data["duration_ms"] = duration_ms
            stage_data["status"] = "success"
            self.stage_metrics.append(stage_data)
            self._log("INFO", f"Stage '{stage_name}' completed ({duration_ms}ms, {stage_data['rows_out']} rows)", 
                     stage=stage_name, duration_ms=duration_ms)
        except Exception as e:
            duration_ms = int((time.time() - stage_start) * 1000)
            stage_data["duration_ms"] = duration_ms
            stage_data["status"] = "failed"
            stage_data["error"] = str(e)
            self.stage_metrics.append(stage_data)
            self._log("ERROR", f"Stage '{stage_name}' FAILED: {e}", stage=stage_name)
            raise
    
    def summary(self):
        """Print pipeline execution summary."""
        total_duration = time.time() - self.start_time
        total_rows = sum(s.get("rows_out", 0) for s in self.stage_metrics)
        failed = sum(1 for s in self.stage_metrics if s["status"] == "failed")
        
        print(f"\n{'='*60}")
        print(f"PIPELINE SUMMARY: {self.pipeline_name}")
        print(f"{'='*60}")
        print(f"Batch: {self.batch_id}")
        print(f"Correlation ID: {self.correlation_id}")
        print(f"Duration: {total_duration:.1f}s")
        print(f"Stages: {len(self.stage_metrics)} ({len(self.stage_metrics)-failed} success, {failed} failed)")
        print(f"Total rows processed: {total_rows:,}")
        print(f"Log entries: {len(self.log_entries)}")
        print(f"{'='*60}")
    
    def write_logs_to_delta(self, table="techmart_dw.pipeline_logs"):
        """Write structured logs to Delta table."""
        if self.log_entries:
            pdf = pd.DataFrame(self.log_entries)
            spark.createDataFrame(pdf).write.format("delta").mode("append").saveAsTable(table)
            print(f"  ✅ {len(self.log_entries)} log entries → {table}")

# Run a real pipeline with production logging
pl = ProductionPipelineLogger("orders_silver_pipeline")

with pl.stage("extract") as s:
    orders = spark.table("techmart_dw.orders")
    s["rows_in"] = orders.count()
    s["rows_out"] = s["rows_in"]

with pl.stage("transform") as s:
    s["rows_in"] = s["rows_out"] = orders.filter("status = 'completed'").count()

with pl.stage("validate") as s:
    valid = orders.filter("order_id IS NOT NULL AND customer_id IS NOT NULL")
    s["rows_in"] = orders.count()
    s["rows_out"] = valid.count()

pl.summary()
pl.write_logs_to_delta()

---
# 📗 Section 9: Databricks-Specific Logging

**Key points:**
- Python `logging` output goes to **driver logs** (Cluster → Driver Logs tab)
- Spark-level logging uses log4j (controlled via `spark.sparkContext.setLogLevel()`)
- UDF logging runs on **workers**, not driver — use accumulators to collect
- In notebooks, `print()` shows in cell output; `logging` goes to driver logs

In [0]:
# Control Spark log verbosity
spark.sparkContext.setLogLevel("WARN")  # Options: ALL, DEBUG, INFO, WARN, ERROR, FATAL, OFF
print("Spark log level set to WARN (reduces noise)")

# In production, you'd set this in cluster config:
# spark.driver.extraJavaOptions=-Dlog4j.configuration=...
# spark.executor.extraJavaOptions=-Dlog4j.configuration=...

---
# 📗 Section 10: The Three Pillars of Observability

```
┌─────────────────────────────────────────────────────────┐
│                    OBSERVABILITY                          │
├──────────────┬──────────────────┬───────────────────────┤
│    LOGS      │     METRICS      │       TRACES          │
│              │                  │                        │
│ What happened│ How much/fast    │ Request flow           │
│ Events       │ Counters/Gauges  │ Correlation IDs        │
│ Errors       │ Time-series      │ Span timing            │
│ Debug info   │ Aggregatable     │ Cross-stage tracking   │
│              │                  │                        │
│ "Stage X     │ "Processed       │ "Request abc123:       │
│  failed at   │  50K rows/min,   │  extract(2s) →         │
│  row 34521"  │  error rate 0.1%"│  transform(5s) →       │
│              │                  │  load(3s) = 10s total" │
└──────────────┴──────────────────┴───────────────────────┘
```

In [0]:
# Simple tracing implementation
import uuid
import time

class PipelineTrace:
    """Simple distributed tracing for multi-stage pipelines."""
    
    def __init__(self, pipeline_name):
        self.trace_id = str(uuid.uuid4())[:12]
        self.pipeline_name = pipeline_name
        self.spans = []
    
    @contextmanager
    def span(self, name):
        span_id = str(uuid.uuid4())[:8]
        start = time.time()
        yield
        duration_ms = int((time.time() - start) * 1000)
        self.spans.append({"span_id": span_id, "name": name, "duration_ms": duration_ms})
    
    def visualize(self):
        total = sum(s["duration_ms"] for s in self.spans)
        print(f"\nTrace: {self.trace_id} ({self.pipeline_name})")
        print(f"Total: {total}ms")
        for s in self.spans:
            bar_len = int(s["duration_ms"] / max(total, 1) * 40)
            bar = "█" * bar_len
            print(f"  {s['name']:<15} {bar} {s['duration_ms']}ms")

# Trace a pipeline
trace = PipelineTrace("orders_etl")

with trace.span("extract"):
    spark.table("techmart_dw.orders").count()

with trace.span("transform"):
    spark.table("techmart_dw.orders").filter("status = 'completed'").count()

with trace.span("load"):
    time.sleep(0.1)  # Simulate write

trace.visualize()

---
# 🏆 Section 11: Interview Questions

## Q1: How do you monitor data pipelines in production?

1. **Structured logging** at each stage (JSON format, correlation IDs)
2. **Metrics collection** (record counts, duration, error rates) → Delta table
3. **Alerting** on thresholds (error rate > 5%, duration > SLA)
4. **Data freshness** monitoring (time since last successful load)
5. **Spark UI** for performance debugging (stages, tasks, shuffles)

## Q2: What metrics do you track?

- `records_processed` — total rows through pipeline
- `processing_time_seconds` — end-to-end duration
- `error_count` / `error_rate_pct` — quality indicator
- `data_freshness_minutes` — time since last load
- `rows_rejected` — quarantined records
- `throughput_records_per_second` — performance indicator

## Q3: How do you handle alert fatigue?

1. **Cooldown periods** — don't re-fire same alert within N minutes
2. **Severity levels** — only page on-call for CRITICAL
3. **Deduplication** — group similar alerts
4. **Escalation** — WARNING → wait → ERROR → page
5. **Auto-resolve** — clear alert when metric returns to normal

## Q4: Logging strategy for multi-step pipeline?

1. Generate `correlation_id` at pipeline start
2. Pass it to every stage (via context/extra fields)
3. Log stage start/end with timing
4. Log record counts (in/out/rejected per stage)
5. Log errors with full context (stage, input sample, stack trace)
6. Write summary at end (total duration, records, error rate)

## Q5: Trace a single record through stages?

Use `correlation_id` + `record_id` in logs. Query logs:
`WHERE correlation_id = 'abc123' ORDER BY timestamp` shows the full journey.

## Q6: Logs vs metrics vs traces?

- **Logs:** Discrete events, text-based, for debugging
- **Metrics:** Numeric time-series, aggregatable, for dashboards
- **Traces:** Request flow across stages, for latency analysis

## Q7: Logging in distributed Spark (driver vs workers)?

- **Driver:** Python `logging` works normally
- **Workers:** UDF logging goes to executor logs (not visible in notebook)
- **Solution:** Use accumulators to collect worker-side counts, or write to a shared table

## Q8: Data freshness SLA violations?

1. Track `last_successful_load_time` per table in a metadata table
2. Scheduled check: `current_time - last_load > SLA_threshold`
3. Alert if violated
4. Dashboard showing all tables with freshness status (green/yellow/red)

---
# 📗 Section 4: Structured Logging for DE Pipelines

Structured logging (JSON format) is the standard in production DE systems.
It enables log aggregation, searching, and alerting in tools like
Datadog, Splunk, CloudWatch, and Databricks system tables.

In [0]:
import logging
import json
import sys
import time
import traceback
from datetime import datetime, timezone
from typing import Any, Dict, Optional

class StructuredFormatter(logging.Formatter):
    """
    Formats log records as JSON for structured logging.
    Compatible with Datadog, Splunk, CloudWatch, etc.
    """
    
    def __init__(self, service_name: str = "de_pipeline",
                 environment: str = "development"):
        super().__init__()
        self.service_name = service_name
        self.environment = environment
    
    def format(self, record: logging.LogRecord) -> str:
        log_entry = {
            "timestamp": datetime.fromtimestamp(
                record.created, tz=timezone.utc
            ).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "service": self.service_name,
            "environment": self.environment,
            "module": record.module,
            "function": record.funcName,
            "line": record.lineno,
        }
        
        # Add exception info if present
        if record.exc_info:
            log_entry["exception"] = {
                "type": record.exc_info[0].__name__,
                "message": str(record.exc_info[1]),
                "traceback": traceback.format_exception(*record.exc_info),
            }
        
        # Add extra fields (from logger.info("msg", extra={...}))
        for key, value in record.__dict__.items():
            if key not in ("name", "msg", "args", "levelname", "levelno",
                           "pathname", "filename", "module", "exc_info",
                           "exc_text", "stack_info", "lineno", "funcName",
                           "created", "msecs", "relativeCreated", "thread",
                           "threadName", "processName", "process", "message"):
                log_entry[key] = value
        
        return json.dumps(log_entry)

def get_pipeline_logger(name: str, level: str = "INFO",
                        service: str = "de_pipeline") -> logging.Logger:
    """Create a structured logger for DE pipelines."""
    logger = logging.getLogger(name)
    logger.setLevel(getattr(logging, level.upper()))
    
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        handler.setFormatter(StructuredFormatter(service_name=service))
        logger.addHandler(handler)
    
    return logger

# Test structured logging
logger = get_pipeline_logger("orders_pipeline", service="techmart_de")

logger.info("Pipeline started", extra={
    "pipeline_id": "orders_daily_2024_01_15",
    "source_table": "bronze.orders",
    "target_table": "silver.orders",
})

logger.warning("Null values detected", extra={
    "column": "customer_id",
    "null_count": 42,
    "total_rows": 10000,
    "null_pct": 0.42,
})

try:
    raise ValueError("Schema mismatch: expected 10 columns, got 9")
except ValueError:
    logger.error("Schema validation failed", exc_info=True, extra={
        "table": "bronze.orders",
        "expected_columns": 10,
        "actual_columns": 9,
    })

In [0]:
# --- Pipeline context manager for automatic logging ---
import contextlib

class PipelineLogger:
    """
    Context manager that logs pipeline start/end/errors automatically.
    Use this to wrap every pipeline stage.
    """
    
    def __init__(self, pipeline_name: str, stage: str,
                 logger: logging.Logger = None, **context):
        self.pipeline_name = pipeline_name
        self.stage = stage
        self.logger = logger or get_pipeline_logger(pipeline_name)
        self.context = context
        self.start_time = None
        self.metrics = {}
    
    def __enter__(self):
        self.start_time = time.time()
        self.logger.info(f"Stage started: {self.stage}", extra={
            "pipeline": self.pipeline_name,
            "stage": self.stage,
            "event": "stage_start",
            **self.context,
        })
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        duration = time.time() - self.start_time
        
        if exc_type is None:
            self.logger.info(f"Stage completed: {self.stage}", extra={
                "pipeline": self.pipeline_name,
                "stage": self.stage,
                "event": "stage_complete",
                "duration_seconds": round(duration, 3),
                **self.metrics,
                **self.context,
            })
        else:
            self.logger.error(f"Stage failed: {self.stage}", extra={
                "pipeline": self.pipeline_name,
                "stage": self.stage,
                "event": "stage_failed",
                "duration_seconds": round(duration, 3),
                "error_type": exc_type.__name__,
                "error_message": str(exc_val),
                **self.context,
            })
        return False  # Don't suppress exceptions
    
    def record(self, **metrics):
        """Record metrics to be included in the completion log."""
        self.metrics.update(metrics)

# Usage example
pipeline_log = get_pipeline_logger("demo_pipeline")

with PipelineLogger("orders_etl", "bronze_ingestion", pipeline_log,
                    source="s3://bucket/orders/") as stage:
    # Simulate work
    time.sleep(0.01)
    stage.record(rows_ingested=15000, files_processed=3, bytes_read=1_500_000)

with PipelineLogger("orders_etl", "silver_transform", pipeline_log) as stage:
    time.sleep(0.01)
    stage.record(rows_in=15000, rows_out=14850, rows_dropped=150)

## 4.2 Log Handlers: File, Rotating, and Remote

Production pipelines write logs to multiple destinations simultaneously.

In [0]:
import logging.handlers
import os
import tempfile

def setup_production_logging(
    logger_name: str,
    log_dir: str = None,
    max_bytes: int = 10 * 1024 * 1024,  # 10 MB
    backup_count: int = 5,
    console_level: str = "INFO",
    file_level: str = "DEBUG",
) -> logging.Logger:
    """
    Production logging setup with:
    - Console handler (INFO+) — for Databricks notebook output
    - Rotating file handler (DEBUG+) — for detailed debugging
    - Structured JSON format for both
    """
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    
    formatter = StructuredFormatter(service_name=logger_name)
    
    # Console handler
    console = logging.StreamHandler(sys.stdout)
    console.setLevel(getattr(logging, console_level.upper()))
    console.setFormatter(formatter)
    logger.addHandler(console)
    
    # Rotating file handler
    if log_dir:
        os.makedirs(log_dir, exist_ok=True)
        log_file = os.path.join(log_dir, f"{logger_name}.log")
        file_handler = logging.handlers.RotatingFileHandler(
            log_file, maxBytes=max_bytes, backupCount=backup_count
        )
        file_handler.setLevel(getattr(logging, file_level.upper()))
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        print(f"  📁 File logging: {log_file}")
    
    return logger

# Test with temp directory
with tempfile.TemporaryDirectory() as tmpdir:
    prod_logger = setup_production_logging(
        "orders_pipeline",
        log_dir=tmpdir,
        console_level="WARNING",  # Only warnings+ to console
        file_level="DEBUG",       # Everything to file
    )
    
    prod_logger.debug("Debug: processing row 1234")
    prod_logger.info("Info: batch complete")
    prod_logger.warning("Warning: 5 nulls found")
    prod_logger.error("Error: connection timeout")
    
    # Read the log file
    log_file = os.path.join(tmpdir, "orders_pipeline.log")
    with open(log_file) as f:
        lines = f.readlines()
    print(f"\nLog file has {len(lines)} entries (all levels):")
    for line in lines:
        entry = json.loads(line)
        print(f"  [{entry['level']}] {entry['message']}")

---
# 📗 Section 5: Metrics & Alerting Patterns

In [0]:
# --- Pipeline metrics collector ---
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List

@dataclass
class PipelineMetrics:
    """Collect and report pipeline execution metrics."""
    pipeline_name: str
    start_time: float = field(default_factory=time.time)
    counters: Dict[str, int] = field(default_factory=lambda: defaultdict(int))
    timings: Dict[str, List[float]] = field(default_factory=lambda: defaultdict(list))
    errors: List[Dict] = field(default_factory=list)
    
    def increment(self, metric: str, value: int = 1):
        self.counters[metric] += value
    
    def time_operation(self, operation: str):
        """Context manager to time an operation."""
        @contextlib.contextmanager
        def _timer():
            start = time.time()
            try:
                yield
            finally:
                elapsed = time.time() - start
                self.timings[operation].append(elapsed)
        return _timer()
    
    def record_error(self, error_type: str, message: str, **context):
        self.errors.append({
            "type": error_type,
            "message": message,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            **context,
        })
        self.increment("errors")
    
    def report(self) -> dict:
        total_time = time.time() - self.start_time
        report = {
            "pipeline": self.pipeline_name,
            "total_duration_seconds": round(total_time, 3),
            "counters": dict(self.counters),
            "timings": {
                op: {
                    "count": len(times),
                    "total_s": round(sum(times), 3),
                    "avg_s": round(sum(times) / len(times), 3),
                    "max_s": round(max(times), 3),
                }
                for op, times in self.timings.items()
            },
            "error_count": len(self.errors),
            "errors": self.errors[:10],  # First 10 errors
        }
        return report

# Simulate a pipeline run
metrics = PipelineMetrics("orders_daily")

# Simulate processing
for i in range(5):
    with metrics.time_operation("db_query"):
        time.sleep(0.001)
    metrics.increment("rows_read", 1000)

metrics.increment("rows_written", 4950)
metrics.increment("rows_dropped", 50)
metrics.record_error("NullValueError", "customer_id is null", row_count=50)

report = metrics.report()
print("Pipeline Metrics Report:")
print(json.dumps(report, indent=2))

---
# 📗 Section 6: Practice Exercises

In [0]:
# ============================================================
# EXERCISE 1: Add Logging to a Data Pipeline
# ============================================================
# Refactor this pipeline to use structured logging

import random

def process_orders_with_logging(orders: list) -> dict:
    """
    Process orders with full structured logging.
    Returns summary metrics.
    """
    logger = get_pipeline_logger("order_processor")
    metrics = PipelineMetrics("order_processor")
    
    logger.info("Starting order processing", extra={
        "input_count": len(orders),
        "event": "pipeline_start",
    })
    
    valid_orders = []
    invalid_orders = []
    
    for order in orders:
        with metrics.time_operation("validate_order"):
            # Validation
            if not order.get("order_id"):
                metrics.record_error("MissingField", "order_id is null",
                                     raw=str(order))
                invalid_orders.append(order)
                continue
            
            if order.get("amount", 0) <= 0:
                metrics.record_error("InvalidAmount",
                                     f"amount={order.get('amount')} is not positive",
                                     order_id=order.get("order_id"))
                invalid_orders.append(order)
                continue
            
            valid_orders.append(order)
            metrics.increment("valid_orders")
    
    metrics.increment("invalid_orders", len(invalid_orders))
    
    logger.info("Processing complete", extra={
        "valid": len(valid_orders),
        "invalid": len(invalid_orders),
        "event": "pipeline_complete",
    })
    
    return metrics.report()

# Test
test_orders = [
    {"order_id": i, "amount": random.choice([100.0, -50.0, 0, 200.0])}
    for i in range(1, 21)
] + [{"amount": 100.0}]  # Missing order_id

result = process_orders_with_logging(test_orders)
print("\nFinal metrics:")
print(f"  Valid: {result['counters'].get('valid_orders', 0)}")
print(f"  Invalid: {result['counters'].get('invalid_orders', 0)}")
print(f"  Errors: {result['error_count']}")

---
# 📗 Section 5: Structured Logging for Data Engineering

## Why Structured Logging?

Plain text logs are hard to query. Structured logs (JSON) are queryable:

```python
# Plain text (hard to parse)
logger.info("Processing 1000 records for date 2024-03-15 in 45.2 seconds")

# Structured (easy to query in Splunk/Datadog/CloudWatch)
logger.info("Pipeline step completed", extra={
    "event": "step_completed",
    "step": "transform_silver",
    "records_processed": 1000,
    "date": "2024-03-15",
    "duration_seconds": 45.2,
    "status": "success"
})
```

## Production Logging Setup

```python
import logging
import json
from datetime import datetime

class StructuredFormatter(logging.Formatter):
    def format(self, record):
        log_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "module": record.module,
            "function": record.funcName,
            "line": record.lineno,
        }
        # Add extra fields
        if hasattr(record, "pipeline_name"):
            log_entry["pipeline_name"] = record.pipeline_name
        if hasattr(record, "records_processed"):
            log_entry["records_processed"] = record.records_processed
        return json.dumps(log_entry)

def setup_pipeline_logger(name, level=logging.INFO):
    logger = logging.getLogger(name)
    logger.setLevel(level)
    handler = logging.StreamHandler()
    handler.setFormatter(StructuredFormatter())
    logger.addHandler(handler)
    return logger
```

## Log Levels -- When to Use Each

| Level | Use For | Example |
|-------|---------|---------|
| DEBUG | Detailed diagnostic info | "Processing record id=42" |
| INFO | Normal operation | "Pipeline started, 1000 records to process" |
| WARNING | Unexpected but handled | "NULL email for customer 42, using default" |
| ERROR | Failure, pipeline continues | "Failed to process batch 3, skipping" |
| CRITICAL | Failure, pipeline stops | "Database connection lost, aborting" |

In [0]:
# Structured logging for data pipelines
import logging
import json
from datetime import datetime

class PipelineLogger:
    """Structured logger for data pipelines."""
    
    def __init__(self, pipeline_name, run_id=None):
        self.pipeline_name = pipeline_name
        self.run_id = run_id or datetime.now().strftime("%Y%m%d_%H%M%S")
        self.logger = logging.getLogger(pipeline_name)
        self.logger.setLevel(logging.DEBUG)
        
        if not self.logger.handlers:
            handler = logging.StreamHandler()
            handler.setFormatter(logging.Formatter("%(message)s"))
            self.logger.addHandler(handler)
        
        self.metrics = {"records_read": 0, "records_written": 0, "errors": 0}
    
    def _log(self, level, message, **kwargs):
        entry = {
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "level": level,
            "pipeline": self.pipeline_name,
            "run_id": self.run_id,
            "message": message,
            **kwargs
        }
        getattr(self.logger, level.lower())(json.dumps(entry))
    
    def info(self, message, **kwargs):
        self._log("INFO", message, **kwargs)
    
    def warning(self, message, **kwargs):
        self._log("WARNING", message, **kwargs)
    
    def error(self, message, **kwargs):
        self._log("ERROR", message, **kwargs)
    
    def step_start(self, step_name):
        self.info(f"Step started: {step_name}", step=step_name, event="step_start")
        return datetime.now()
    
    def step_end(self, step_name, start_time, records=None):
        duration = (datetime.now() - start_time).total_seconds()
        self.info(f"Step completed: {step_name}",
                 step=step_name, event="step_end",
                 duration_seconds=round(duration, 2),
                 records_processed=records)
    
    def quality_check(self, check_name, passed, failures=0):
        level = "INFO" if passed else "WARNING"
        self._log(level, f"Quality check: {check_name}",
                 check=check_name, passed=passed, failures=failures,
                 event="quality_check")


# Demo
logger = PipelineLogger("daily_revenue_pipeline")

logger.info("Pipeline started", date="2024-03-15", source="techmart_dw.orders")

t = logger.step_start("ingest_bronze")
logger.step_end("ingest_bronze", t, records=20000)

t = logger.step_start("transform_silver")
logger.warning("NULL customer_ids found", null_count=42, action="quarantined")
logger.step_end("transform_silver", t, records=19958)

logger.quality_check("no_null_order_ids", passed=True)
logger.quality_check("positive_amounts", passed=False, failures=3)

t = logger.step_start("build_gold")
logger.step_end("build_gold", t, records=365)

logger.info("Pipeline completed", total_duration_seconds=127.4, status="success")


---
# ✅ Notebook Complete!

**What was covered:**
- Python logging module: loggers, handlers, formatters, levels
- Structured logging: JSON format, correlation IDs, extra fields
- Pipeline patterns: stage context managers, record count tracking
- Log levels strategy with real scenarios
- Monitoring: PipelineMetrics class writing to Delta
- Alerting: threshold checks with cooldown logic
- Mini project: ProductionPipelineLogger with full observability
- Databricks-specific: driver logs, Spark log levels
- Three pillars: logs + metrics + traces with visualization
- 8 interview questions

**Next:** Notebook 14 — Datetime for Data Engineering

---